<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle%20Study/10__Chest_X_ray_Images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tolgadincer/labeled-chest-xray-images")

print("Path to dataset files:", path)

100%|██████████| 1.17G/1.17G [00:11<00:00, 113MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/tolgadincer/labeled-chest-xray-images/versions/1


In [ ]:
!ls -al /root/.cache/kagglehub/datasets/tolgadincer/labeled-chest-xray-images/versions/1/chest_xray/train
# !ls -al /kaggle/input/labeled-chest-xray-images/chest_xray/train

total 276
drwxr-xr-x 4 root root   4096 Jun  3 10:49 .
drwxr-xr-x 4 root root   4096 Jun  3 10:49 ..
drwxr-xr-x 2 root root  69632 Jun  3 10:49 NORMAL
drwxr-xr-x 2 root root 196608 Jun  3 10:49 PNEUMONIA


In [ ]:
!ls -al /root/.cache/kagglehub/datasets/tolgadincer/labeled-chest-xray-images/versions/1/train/NORMAL

ls: cannot access '/root/.cache/kagglehub/datasets/tolgadincer/labeled-chest-xray-images/versions/1/train/NORMAL': No such file or directory


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

N_SPLITS = 5
path = '/root/.cache/kagglehub/datasets/tolgadincer/labeled-chest-xray-images/versions/1/chest_xray/'
# path = '/kaggle/input/labeled-chest-xray-images/chest_xray/'

train_dir = os.path.join(path, 'train')
test_dir = os.path.join(path, 'test')
categories = ['NORMAL', 'PNEUMONIA']

train_val_data = []
for category in categories:
    folder_path = os.path.join(train_dir, category)
    for img_name in os.listdir(folder_path):
        if img_name.endswith(('jpg', 'jpeg', 'png')):
            train_val_data.append({
                'image_path': os.path.join(folder_path, img_name),
                'label_name': category
            })

train_val_data_frame = pd.DataFrame(train_val_data)

test_data = []
for category in categories:
    folder_path = os.path.join(test_dir, category)
    for img_name in os.listdir(folder_path):
        if img_name.endswith(('jpg', 'jpeg', 'png')):
            test_data.append({
                'image_path': os.path.join(folder_path, img_name),
                'label_name': category
            })

test_data_frame = pd.DataFrame(test_data)

label_to_idx = {label_name: idx for idx, label_name in enumerate(categories)}
train_val_data_frame['label'] = train_val_data_frame['label_name'].map(label_to_idx)
test_data_frame['label'] = test_data_frame['label_name'].map(label_to_idx)

# train_val_df, test_df = train_test_split(
#     df, test_size=0.15, random_state=42, stratify=df['label']
# )

# train_val_df = train_val_df.reset_index(drop=True)
# test_df = test_df.reset_index(drop=True)

train_val_data_frame['fold'] = -1
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold, (train_index, val_index) in enumerate(skf.split(train_val_data_frame, train_val_data_frame['label'])):
    train_val_data_frame.loc[val_index, 'fold'] = fold

print(f"Train/Val 데이터 개수: {len(train_val_data_frame)}, Test 데이터 개수: {len(test_data_frame)}")

Train/Val 데이터 개수: 5232, Test 데이터 개수: 624


In [ ]:
import cv2
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class ChestXrayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = int(row['label'])

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(label, dtype=torch.long)

train_transform = A.Compose([
    A.Resize(128, 128),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
from torch.utils.data import DataLoader
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epochs = 3
BATCH_SIZE = 128

test_dataset = ChestXrayDataset(df=test_data_frame, transform=val_test_transform)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
total_test_preds = torch.zeros(len(test_data_frame), len(categories)).to(device)

for fold in range(N_SPLITS):
    train_data = train_val_data_frame[train_val_data_frame['fold'] != fold].reset_index(drop=True)
    val_data = train_val_data_frame[train_val_data_frame['fold'] == fold].reset_index(drop=True)

    train_dataset = ChestXrayDataset(df=train_data, transform=train_transform)
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    val_dataset = ChestXrayDataset(df=val_data, transform=val_test_transform)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = timm.create_model('resnet18', pretrained=True, num_classes=len(categories)).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            train_correct += (preds == labels).sum().item()

        epoch_train_loss = train_loss / len(train_data)
        epoch_train_acc = train_correct / len(train_data) * 100
        print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

        model.eval()
        val_loss = 0.0
        val_correct = 0

        with torch.no_grad():
            for images, labels in val_dataloader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                val_correct += (preds == labels).sum().item()

        epoch_val_loss = val_loss / len(val_data)
        epoch_val_acc = val_correct / len(val_data) * 100
        print(f"Epoch {epoch+1} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), f"best_model_fold{fold}.pth")

    print(f"--> Load Best Model of Fold {fold} and Predict Test Data")
    best_model = timm.create_model('resnet18', pretrained=True, num_classes=len(categories)).to(device)
    best_model.load_state_dict(torch.load(f"best_model_fold{fold}.pth", map_location=device))
    best_model.eval()

    fold_test_preds = []
    with torch.no_grad():
        for images, _ in test_dataloader:
            images = images.to(device)
            outputs = best_model(images)

            probs = outputs.softmax(1)
            fold_test_preds.append(probs)

    total_test_preds += torch.cat(fold_test_preds, 0)

final_test_preds = total_test_preds / N_SPLITS
final_classes = final_test_preds.argmax(1).cpu().numpy()

test_data_frame['predicted_label'] = final_classes
test_data_frame.to_csv('submission.csv', index=False)

Epoch 1 | Train Loss: 0.1860 | Train Acc: 93.24%
Epoch 1 | Val Loss: 0.0562 | Val Acc: 98.19%
Epoch 2 | Train Loss: 0.0565 | Train Acc: 97.90%
Epoch 2 | Val Loss: 0.1716 | Val Acc: 93.03%
Epoch 3 | Train Loss: 0.0311 | Train Acc: 98.81%
Epoch 3 | Val Loss: 0.0361 | Val Acc: 98.76%
--> Load Best Model of Fold 0 and Predict Test Data
Epoch 1 | Train Loss: 0.2153 | Train Acc: 91.11%
Epoch 1 | Val Loss: 0.4927 | Val Acc: 82.14%
Epoch 2 | Train Loss: 0.0557 | Train Acc: 97.87%
Epoch 2 | Val Loss: 0.1525 | Val Acc: 94.56%
Epoch 3 | Train Loss: 0.0340 | Train Acc: 98.83%
Epoch 3 | Val Loss: 0.0710 | Val Acc: 97.23%
--> Load Best Model of Fold 1 and Predict Test Data
Epoch 1 | Train Loss: 0.1752 | Train Acc: 93.79%
Epoch 1 | Val Loss: 0.0724 | Val Acc: 96.85%
Epoch 2 | Train Loss: 0.0505 | Train Acc: 98.11%
Epoch 2 | Val Loss: 0.0852 | Val Acc: 96.65%
Epoch 3 | Train Loss: 0.0242 | Train Acc: 99.12%
Epoch 3 | Val Loss: 0.0404 | Val Acc: 98.47%
--> Load Best Model of Fold 2 and Predict Test Dat